# Why naive A/B tests break in delivery marketplaces

**Authors:** Brandon Kim, Kentaro Davies

This notebook walks through the project one step at a time:

1. Build a small simulated marketplace of customers, Dashers and restaurants.
2. Test a **$1 delivery-fee cut** with a standard user-level A/B test and show that it's biased.
3. Fix it with a **switchback experiment** that randomizes whole region × 2-hour blocks.
4. Tighten the switchback's noisy estimate with **CUPED**.

The full study, with many repeated simulations, is in `run_experiments.py`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # so `marketsim` imports when run from notebooks/

import numpy as np
import matplotlib.pyplot as plt
from marketsim.market import MarketConfig, make_regions, make_world, simulate, PERIODS_PER_DAY
from marketsim.experiments import ground_truth, naive_ab, switchback, pre_period_covariates, ci

## 1. The market

20 regions, 14 days, 30-minute periods. Customers convert less when fees or ETAs are higher.
ETAs rise sharply as orders approach Dasher capacity, and that's the link that creates interference.

In [ ]:
cfg = MarketConfig()
rng = np.random.default_rng(7)
regions = make_regions(cfg, rng)
pre_world = make_world(cfg, regions, rng, start_day=0)        # 2 weeks before the test
world = make_world(cfg, regions, rng, start_day=cfg.days)     # the 2-week test
T, R = world.sessions.shape
print(f"{world.sessions.sum():,} app sessions, {R} regions, {T} periods")

hour = (np.arange(T) % PERIODS_PER_DAY) / 2
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(np.unique(hour), [world.sessions[hour == h].mean() for h in np.unique(hour)], label="sessions")
ax.plot(np.unique(hour), [world.dashers[hour == h].mean() for h in np.unique(hour)], label="Dashers")
ax.set_xlabel("hour of day"); ax.set_title("Demand peaks faster than Dasher supply"); ax.legend();

## 2. Ground truth

What really happens if the fee cut ships to *everyone*, compared with *no one*, in the same market.
A perfect experiment would recover this number.

In [ ]:
truth = ground_truth(world)
for m in ("conversion", "revenue"):
    print(f"{m:>10}: true lift {truth[m]['lift']:+.1%}")
print(f"ETA: {truth['eta_control']:.1f} min -> {truth['eta_treatment']:.1f} min with the fee cut")

## 3. The naive A/B test

Half the customers in every region see the lower fee. Both halves share the same Dashers.
The treatment group's extra orders slow down deliveries for the control group too, so control looks
worse than it really would be, and the gap between the groups is inflated.

In [ ]:
naive = naive_ab(world, rng)
for m in ("conversion", "revenue"):
    n = naive[m]; lo, hi = ci(n["effect"], n["se"])
    print(f"{m:>10}: naive lift {n['lift']:+.1%}  (95% CI {lo / n['control']:+.1%} to {hi / n['control']:+.1%})"
          f"   vs truth {truth[m]['lift']:+.1%}")

Note the revenue line: the A/B test says the fee cut **makes** money, but rolled out to everyone it **loses** money.
The confidence interval is tight, so the test is confidently wrong. More data doesn't fix bias.

## 4. Switchback + CUPED

Each region's two weeks are cut into 2-hour blocks, and each block is entirely treatment or entirely
control. Within a block nobody competes with the other arm for Dashers, so the comparison matches a real rollout.

There are far fewer independent units (region-blocks) than customers, so we need **cluster-robust standard
errors** (rows in the same block are correlated) and **CUPED** to claw back precision. The CUPED covariate
is each region's outcome in the same weekly time slot during the two pre-test weeks.

In [ ]:
pre_out = simulate(pre_world, np.zeros((T, R)), rng)
cov = pre_period_covariates(pre_out, T, block_periods=4)
sb = switchback(world, rng, block_periods=4, burn_in=0, pre_covariates=cov)

for m in ("conversion", "revenue"):
    s = sb[m]
    print(f"{m:>10}: switchback {s['lift']:+.1%} (SE {s['se'] / s['control']:.2%})   "
          f"+CUPED {s['lift_cuped']:+.1%} (SE {s['se_cuped'] / s['control']:.2%})   truth {truth[m]['lift']:+.1%}")
print(f"\nPlain SE that ignores clustering: {sb['conversion']['se_iid']:.4f}  vs cluster-robust: {sb['conversion']['se']:.4f}")

## 5. Takeaways

_One switchback is a single noisy draw, so it can land a couple of standard errors from the truth. The naive test misses by about 12 standard errors, every time. `run_experiments.py` repeats this 40 times to show the switchback is unbiased on average._

- **Naive A/B tests in a shared-supply marketplace are biased, not just noisy.** Here a fee cut looks about 3x better than it is, and its revenue effect flips sign.
- **Switchbacks remove the interference** by making every competing customer in a region and time block see the same treatment.
- **The cost is power.** Switchbacks have few units. Cluster-robust SEs keep the confidence intervals honest, and CUPED wins back roughly a third of the variance.

See `results/summary.md` for the results over 40 simulated markets.